In [27]:
import requests
from lxml import etree
from bs4 import BeautifulSoup

# Block 1: Fetch All API Entries
def fetch_all_api_entries(url):
    """Fetch all available entries from the SyncFeed API."""
    entries = []
    while url:
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"Failed to fetch API data: {e}")
            break
        
        root = etree.fromstring(response.content)
        entries.extend(root.findall('.//{http://www.w3.org/2005/Atom}entry'))
        next_link = root.find('.//{http://www.w3.org/2005/Atom}link[@rel="next"]')
        url = next_link.get('href') if next_link is not None else None
    
    return entries

In [28]:
# Block 2: Extract Topics
def extract_topics(entries):
    """Extract meaningful topics from API entries, with debugging."""
    topics = []
    for entry in entries:
        title = entry.find('.//{http://www.w3.org/2005/Atom}title')
        summary = entry.find('.//{http://www.w3.org/2005/Atom}summary')
        content = entry.find('.//{http://www.w3.org/2005/Atom}content')
        
        # Debugging output
        print(f"Entry ID: {entry.find('.//{http://www.w3.org/2005/Atom}id')}")
        print(f"Title: {title.text if title is not None else 'None'}")
        print(f"Summary: {summary.text if summary is not None else 'None'}")
        print(f"Content: {content.text if content is not None else 'None'}")
        print("---")
        
        # Use content or summary if available, fallback to title
        if content is not None and content.text and "Motie" in content.text:
            topics.append(content.text.strip())
        elif summary is not None and summary.text and "Motie" in summary.text:
            topics.append(summary.text.strip())
        elif title is not None and title.text:
            topics.append(title.text.strip())  # Fallback to UUID if no better data
    
    return topics

In [29]:
# Block 3: Scrape Web Page
def scrape_web_page(url, topics):
    """Scrape the web page for motions matching the given topics."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch web page: {e}")
        return []
    
    soup = BeautifulSoup(response.text, 'html.parser')
    sections = []
    
    # Find all <h3> tags with "Stemmingen" to locate voting sections
    voting_headers = soup.find_all('h3', class_='m-timeline__title', string=lambda t: "Stemmingen" in t if t else False)
    for header in voting_headers:
        # Get the parent timeline item and find all toggler elements (motions)
        timeline_item = header.find_parent('li', class_='m-timeline__item')
        if not timeline_item:
            continue
        
        togglers = timeline_item.find_all('div', class_='m-toggler')
        for toggler in togglers:
            label = toggler.find('span', class_='m-toggler__label')
            if label and label.text:
                motion_text = label.text.strip()
                # Check if any topic (or part of it) matches the motion text
                for topic in topics:
                    if topic in motion_text or any(word in motion_text for word in topic.split() if len(word) > 3):
                        details = toggler.find('div', class_='m-toggler__body')
                        detail_text = details.get_text(strip=True) if details else "Details not loaded (AJAX)"
                        sections.append((motion_text, detail_text))
                        break  # Move to next motion once matched
    
    return sections

In [ ]:
# Main Execution
def main():
    api_url = "https://gegevensmagazijn.tweedekamer.nl/SyncFeed/2.0/Feed?category=Stemming"
    web_page_url = "https://www.tweedekamer.nl/debat_en_vergadering/plenaire_vergaderingen/details/activiteit?id=2025A01853"
    
    # Step 1: Fetch all entries from API
    print("Fetching all API entries...")
    all_entries = fetch_all_api_entries(api_url)
    if not all_entries:
        print("No entries fetched from API.")
        return
    print(f"Total entries fetched: {len(all_entries)}")
    
    # Step 2: Select first 3 entries
    print("Selecting first 3 entries...")
    first_ten_entries = all_entries[:3]
    if not first_ten_entries:
        print("No entries available to select.")
        return
    
    # Step 3: Extract topics from the first 3 entries
    print("Extracting topics from first 10 entries...")
    topics = extract_topics(first_ten_entries)
    if not topics:
        print("No topics extracted.")
        return
    print(f"Found {len(topics)} topics: {topics}")
    
    # Step 4: Scrape web page using the topics
    print("Scraping web page with selected topics...")
    sections = scrape_web_page(web_page_url, topics)
    if not sections:
        print("No matching sections found on web page.")
        return
    
    # Step 5: Display results
    print("\nResults:")
    for topic, detail in sections:
        print(f"Topic: {topic}")
        print(f"Detail: {detail}")
        print("---")

if __name__ == "__main__":
    main()